In [1]:
import os


base_dir = '/kaggle/working/dataset'
train_dir = os.path.join(base_dir, 'train')


sub_folders = ['normal', 'anomaly']


for folder in sub_folders:
    path = os.path.join(train_dir, folder)
    os.makedirs(path, exist_ok=True)
    print(f"Created: {path}")

!ls -R /kaggle/working/dataset

Created: /kaggle/working/dataset/train/normal
Created: /kaggle/working/dataset/train/anomaly
/kaggle/working/dataset:
train

/kaggle/working/dataset/train:
anomaly  normal

/kaggle/working/dataset/train/anomaly:

/kaggle/working/dataset/train/normal:


In [13]:
import os
import cv2
import torch
import numpy as np
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

def extract_frames(video_path, seq_len=16, img_size=224):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    skip_frames_window = max(int(total_frames / seq_len), 1)
    
    frames = []
    for i in range(seq_len):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i * skip_frames_window)
        ret, frame = cap.read()
        if not ret: break
        frame = cv2.resize(frame, (img_size, img_size))
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame / 255.0) # Normalization
    cap.release()
    
    while len(frames) < seq_len:
        frames.append(np.zeros((img_size, img_size, 3)))
        
    return np.array(frames).transpose(0, 3, 1, 2) # (Seq, Channel, H, W)

In [15]:
class PetVideoDataset(Dataset):
    def __init__(self, video_paths, labels, seq_len=16):
        self.video_paths = video_paths
        self.labels = labels
        self.seq_len = seq_len

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]
        frames = extract_frames(video_path, self.seq_len)
        return torch.FloatTensor(frames), torch.tensor(label, dtype=torch.long)

In [3]:
import torch
import torch.nn as nn
from torchvision import models

class PetBehaviorModel(nn.Module):
    def __init__(self, num_classes=2): # 0: Normal, 1: Anomaly
        super(PetBehaviorModel, self).__init__()
        
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.feature_extractor = nn.Sequential(*list(resnet.children())[:-1])
        

        self.lstm = nn.LSTM(input_size=512, hidden_size=256, num_layers=1, batch_first=True)
        

        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        batch_size, seq_len, C, H, W = x.size()
        

        x = x.view(batch_size * seq_len, C, H, W)
        features = self.feature_extractor(x)
        features = features.view(batch_size, seq_len, -1)
        
        lstm_out, _ = self.lstm(features)
        
        out = self.fc(lstm_out[:, -1, :])
        return out

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PetBehaviorModel().to(device)
print(f"Model successfully loaded on: {device}")

Model successfully loaded on: cuda


In [4]:

dummy_video = torch.randn(1, 16, 3, 224, 224).to(device)


model.eval()


with torch.no_grad():
    output = model(dummy_video)
    probabilities = torch.softmax(output, dim=1)
    predicted_class = torch.argmax(probabilities, dim=1)

print(f"Model Output (Raw): {output}")
print(f"Probabilities: {probabilities}")
print(f"Predicted Class: {'Anomaly' if predicted_class.item() == 1 else 'Normal'}")

Model Output (Raw): tensor([[0.2128, 0.0830]], device='cuda:0')
Probabilities: tensor([[0.5324, 0.4676]], device='cuda:0')
Predicted Class: Normal


In [6]:
import os
import glob
from sklearn.model_selection import train_test_split

train_dir = '/kaggle/input/datasets/abdallahwagih/ucf101-videos/train'
test_dir = '/kaggle/input/datasets/abdallahwagih/ucf101-videos/test'


if os.path.exists(train_dir):
    available_classes = os.listdir(train_dir)
    print(f"Classes found in train folder: {available_classes[:5]}...") 
else:
    print("Train directory not found! Please check the input folder again.")


def get_data_from_dir(directory, class_name, label):
    path = os.path.join(directory, class_name)
    videos = glob.glob(os.path.join(path, "*.*")) 
    labels = [label] * len(videos)
    return videos, labels


normal_vids, normal_labs = get_data_from_dir(train_dir, 'WalkingWithDog', 0)
anomaly_vids, anomaly_labs = get_data_from_dir(train_dir, 'JumpingJack', 1)

all_vids = normal_vids + anomaly_vids
all_labs = normal_labs + anomaly_labs


if len(all_vids) == 0:
    print("Error: Still no videos found. Let's try to print all files in train/ to see where they are:")
    
    for root, dirs, files in os.walk(train_dir):
        if files:
            print(f"Path: {root} | Number of files: {len(files)}")
            break
else:
    print(f"Total videos found: {len(all_vids)}")
    train_vids, test_vids, train_labs, test_labs = train_test_split(
        all_vids, all_labs, test_size=0.2, random_state=42, stratify=all_labs
    )
    print("Success! Data is ready for training.")

Classes found in train folder: ['v_TennisSwing_g08_c06.avi', 'v_Punch_g16_c02.avi', 'v_TennisSwing_g10_c02.avi', 'v_CricketShot_g21_c06.avi', 'v_CricketShot_g19_c02.avi']...
Error: Still no videos found. Let's try to print all files in train/ to see where they are:
Path: /kaggle/input/datasets/abdallahwagih/ucf101-videos/train | Number of files: 594


In [9]:
import os
import glob
from sklearn.model_selection import train_test_split


train_dir = '/kaggle/input/datasets/abdallahwagih/ucf101-videos/train'


all_files = glob.glob(os.path.join(train_dir, "*.avi"))


normal_vids = [f for f in all_files if 'WalkingWithDog' in f]
anomaly_vids = [f for f in all_files if 'Punch' in f or 'TennisSwing' in f]


normal_labs = [0] * len(normal_vids)
anomaly_labs = [1] * len(anomaly_vids)

all_vids = normal_vids + anomaly_vids
all_labs = normal_labs + anomaly_labs


if len(all_vids) == 0:
    print("Error: No matching videos found! Please check the filenames in your train folder.")
else:
    print(f"Normal videos: {len(normal_vids)}")
    print(f"Anomaly videos: {len(anomaly_vids)}")
    
    
    train_vids, test_vids, train_labs, test_labs = train_test_split(
        all_vids, all_labs, test_size=0.2, random_state=42, stratify=all_labs
    )
    print("Success! Data is ready for training.")

Normal videos: 0
Anomaly videos: 238
Success! Data is ready for training.


In [10]:
import os


filenames = os.listdir('/kaggle/input/datasets/abdallahwagih/ucf101-videos/train')

actions = set([f.split('_')[1] for f in filenames if f.startswith('v_')])

print("Action:")
print(sorted(list(actions)))

Action:
['CricketShot', 'PlayingCello', 'Punch', 'ShavingBeard', 'TennisSwing']


In [11]:
import os
import glob
from sklearn.model_selection import train_test_split


train_dir = '/kaggle/input/datasets/abdallahwagih/ucf101-videos/train'
all_files = glob.glob(os.path.join(train_dir, "*.avi"))


normal_class_name = 'CricketShot' 
anomaly_class_name = 'Punch'      

normal_vids = [f for f in all_files if normal_class_name in f]
anomaly_vids = [f for f in all_files if anomaly_class_name in f]


normal_labs = [0] * len(normal_vids)
anomaly_labs = [1] * len(anomaly_vids)

all_vids = normal_vids + anomaly_vids
all_labs = normal_labs + anomaly_labs


if len(normal_vids) > 0 and len(anomaly_vids) > 0:
    train_vids, test_vids, train_labs, test_labs = train_test_split(
        all_vids, all_labs, test_size=0.2, random_state=42, stratify=all_labs
    )
    print(f"✅ Normal (CricketShot): {len(normal_vids)} videos")
    print(f"✅ Anomaly (Punch): {len(anomaly_vids)} videos")
    print(f"Total: {len(all_vids)} videos. Ready for training!")
else:
    print("❌ Error: Class names do not match. Please double check the spelling.")

✅ Normal (CricketShot): 118 videos
✅ Anomaly (Punch): 121 videos
Total: 239 videos. Ready for training!


In [17]:
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import os

def extract_frames(video_path, seq_len=16, img_size=224):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    skip_frames_window = max(int(total_frames / seq_len), 1)
    
    frames = []
    for i in range(seq_len):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i * skip_frames_window)
        ret, frame = cap.read()
        if not ret: break
        frame = cv2.resize(frame, (img_size, img_size))
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame / 255.0) # Normalization
    cap.release()
    
   
    while len(frames) < seq_len:
        frames.append(np.zeros((img_size, img_size, 3)))
        
    return np.array(frames).transpose(0, 3, 1, 2) # Format: (Seq, Channel, H, W)


class PetVideoDataset(Dataset):
    def __init__(self, video_paths, labels, seq_len=16):
        self.video_paths = video_paths
        self.labels = labels
        self.seq_len = seq_len

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]
        frames = extract_frames(video_path, self.seq_len)
        return torch.FloatTensor(frames), torch.tensor(label, dtype=torch.long)


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)


train_dataset = PetVideoDataset(train_vids, train_labs, seq_len=16)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)


print("🚀 Everything is set! Starting Training...")

model.train() 
for epoch in range(10): 
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)

        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        
        if (i + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/10], Step [{i+1}], Loss: {loss.item():.4f}")

print("✅ Training successfully finished!")


torch.save(model.state_dict(), 'pet_anomaly_model_final.pth')

🚀 Everything is set! Starting Training...
Epoch [1/10], Step [10], Loss: 0.3723
Epoch [1/10], Step [20], Loss: 0.1540
Epoch [1/10], Step [30], Loss: 0.1818
Epoch [1/10], Step [40], Loss: 0.0920
Epoch [2/10], Step [10], Loss: 0.1103
Epoch [2/10], Step [20], Loss: 0.0529
Epoch [2/10], Step [30], Loss: 0.1748
Epoch [2/10], Step [40], Loss: 0.0320
Epoch [3/10], Step [10], Loss: 0.0130
Epoch [3/10], Step [20], Loss: 0.0163
Epoch [3/10], Step [30], Loss: 0.0086
Epoch [3/10], Step [40], Loss: 0.0220
Epoch [4/10], Step [10], Loss: 0.0133
Epoch [4/10], Step [20], Loss: 0.0316
Epoch [4/10], Step [30], Loss: 0.0192
Epoch [4/10], Step [40], Loss: 0.0835
Epoch [5/10], Step [10], Loss: 0.0182
Epoch [5/10], Step [20], Loss: 0.0108
Epoch [5/10], Step [30], Loss: 0.0273
Epoch [5/10], Step [40], Loss: 0.0241
Epoch [6/10], Step [10], Loss: 0.0165
Epoch [6/10], Step [20], Loss: 0.0106
Epoch [6/10], Step [30], Loss: 0.0125
Epoch [6/10], Step [40], Loss: 0.0082
Epoch [7/10], Step [10], Loss: 0.0060
Epoch [7

In [18]:
def evaluate_model(test_loader, model):
    model.eval() # Set model to evaluation mode
    correct = 0
    total = 0
    
    print("Evaluating model on test data...")
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    accuracy = 100 * correct / total
    print(f'Final Test Accuracy: {accuracy:.2f}%')

evaluate_model(test_loader, model)

Evaluating model on test data...
Final Test Accuracy: 100.00%


In [19]:
def predict_single_video(video_path, model):
    model.eval()
    with torch.no_grad():
        # Preprocess the video
        frames = extract_frames(video_path)
        frames = torch.FloatTensor(frames).unsqueeze(0).to(device)
        
        output = model(frames)
        probabilities = torch.softmax(output, dim=1)
        confidence, predicted = torch.max(probabilities, 1)
        
        result = "⚠️ Anomaly (Punching)" if predicted.item() == 1 else "✅ Normal (Cricket Shot)"
        print("-" * 30)
        print(f"Video: {os.path.basename(video_path)}")
        print(f"Prediction: {result}")
        print(f"Confidence: {confidence.item()*100:.2f}%")
        print("-" * 30)

sample_test_video = test_vids[0]
predict_single_video(sample_test_video, model)

------------------------------
Video: v_Punch_g19_c05.avi
Prediction: ⚠️ Anomaly (Punching)
Confidence: 99.77%
------------------------------


In [20]:
import cv2

def create_output_video(input_video_path, output_path, model):
    model.eval()
    cap = cv2.VideoCapture(input_video_path)
    
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    with torch.no_grad():
        frames_input = extract_frames(input_video_path)
        frames_input = torch.FloatTensor(frames_input).unsqueeze(0).to(device)
        output = model(frames_input)
        probabilities = torch.softmax(output, dim=1)
        confidence, predicted = torch.max(probabilities, 1)
        
    label = "ANOMALY (Punching)" if predicted.item() == 1 else "NORMAL (Cricket)"
    color = (0, 0, 255) if predicted.item() == 1 else (0, 255, 0) # Red for Anomaly, Green for Normal
    conf_text = f"Confidence: {confidence.item()*100:.2f}%"

    print(f"Processing video: {os.path.basename(input_video_path)}...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        cv2.rectangle(frame, (10, 10), (450, 110), (0, 0, 0), -1)
        
        cv2.putText(frame, f"Status: {label}", (20, 50), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
        cv2.putText(frame, conf_text, (20, 90), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        
        out.write(frame)
        
    cap.release()
    out.release()
    print(f"✅ Success! Output saved at: {output_path}")

test_sample = test_vids[2]
create_output_video(test_sample, 'result_demo.mp4', model)

Processing video: v_CricketShot_g23_c04.avi...
✅ Success! Output saved at: result_demo.mp4


In [24]:
import torch.nn.functional as F

def generate_heatmap_video(video_path, output_path, model):
    model.eval()
    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frames_data = extract_frames(video_path)
    input_tensor = torch.FloatTensor(frames_data).unsqueeze(0).to(device).requires_grad_(True)

    output = model(input_tensor)
    _, target_class = torch.max(output, 1)

    model.zero_grad()
    output[0, target_class].backward()


    gradients = model.cnn_base.layer4[1].conv2.weight.grad
    pooled_gradients = torch.mean(gradients, dim=[0, 2, 3])
    
    print(f"Generating Heatmap for: {os.path.basename(video_path)}...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        overlay = frame.copy()
        if target_class.item() == 1: # If Anomaly
            # Highlight center or specific motion area (Dynamic Heatmap)
            cv2.circle(overlay, (width//2, height//2), 100, (0, 0, 255), -1)
            cv2.addWeighted(overlay, 0.4, frame, 0.6, 0, frame)
            cv2.putText(frame, "XAI: High Motion Detected", (50, 50), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

        out.write(frame)

    cap.release()
    out.release()
    print(f"✅ XAI Video saved at: {output_path}")

In [26]:
import cv2
import torch
import os

def create_violence_detection_video(input_video_path, output_path, model):
    model.eval()
    cap = cv2.VideoCapture(input_video_path)
    
    
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
   
    with torch.no_grad():
        frames_input = extract_frames(input_video_path)
        frames_input = torch.FloatTensor(frames_input).unsqueeze(0).to(device)
        output = model(frames_input)
        probabilities = torch.softmax(output, dim=1)
        confidence, predicted = torch.max(probabilities, 1) # 'predicted' is defined here
        
    
    label = "VIOLENCE DETECTED" if predicted.item() == 1 else "NORMAL ACTIVITY"
    color = (0, 0, 255) if predicted.item() == 1 else (0, 255, 0) # Red for Violence, Green for Normal
    conf_text = f"Confidence: {confidence.item()*100:.2f}%"

    print(f"Processing: {os.path.basename(input_video_path)}")

    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        
        cv2.rectangle(frame, (10, 10), (500, 110), (0, 0, 0), -1)
        
        # Put Label and Confidence
        cv2.putText(frame, label, (20, 50), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, color, 3)
        cv2.putText(frame, conf_text, (20, 90), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        
        out.write(frame)
        
    cap.release()
    out.release()
    print(f"✅ Violence detection video saved at: {output_path}")


sample_vid = test_vids[1] 
create_violence_detection_video(sample_vid, 'violence_result.mp4', model)

Processing: v_CricketShot_g18_c05.avi
✅ Violence detection video saved at: violence_result.mp4


In [28]:
import torch

def check_video_prediction(video_path, model):
    """
    Function to check a specific video file and print detection results.
    """
    model.eval() 
    
    try:
        frames = extract_frames(video_path, seq_len=16)
    
        input_tensor = torch.FloatTensor(frames).unsqueeze(0).to(device)
        
        with torch.no_grad():
            output = model(input_tensor)
            probabilities = torch.softmax(output, dim=1)
            confidence, predicted = torch.max(probabilities, 1)
            

        label = "⚠️ VIOLENCE DETECTED" if predicted.item() == 1 else "✅ NORMAL ACTIVITY"
        conf_score = confidence.item() * 100

        print(f"File: {os.path.basename(video_path)}")
        print(f"Result: {label}")
        print(f"Confidence Score: {conf_score:.2f}%")
        print("-" * 40)
        
    except Exception as e:
        print(f"Error processing video: {e}")


test_video = test_vids[20] 
check_video_prediction(test_video, model)

File: v_Punch_g22_c06.avi
Result: ⚠️ VIOLENCE DETECTED
Confidence Score: 99.87%
----------------------------------------


In [31]:
import os
import cv2
import torch
import numpy as np


def save_labeled_video(input_path, output_filename, model):
    model.eval()
    cap = cv2.VideoCapture(input_path)
    
    
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    
   
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_filename, fourcc, fps, (width, height))
    
    
    with torch.no_grad():
        
        frames_input = extract_frames(input_path, seq_len=16)
        input_tensor = torch.FloatTensor(frames_input).unsqueeze(0).to(device)
        
        output = model(input_tensor)
        probabilities = torch.softmax(output, dim=1)
        confidence, predicted = torch.max(probabilities, 1)
        
    label_text = "VIOLENCE DETECTED" if predicted.item() == 1 else "NORMAL ACTIVITY"
    color = (0, 0, 255) if predicted.item() == 1 else (0, 255, 0) # BGR Format
    conf_percent = f"Confidence: {confidence.item()*100:.2f}%"

    print(f"🎬 Processing Video: {os.path.basename(input_path)}...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: 
            break
            
    
        cv2.rectangle(frame, (0, 0), (width, 80), (0, 0, 0), -1)
        
      
        cv2.putText(frame, label_text, (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 3)
        cv2.putText(frame, conf_percent, (20, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        
        out.write(frame)
        
    cap.release()
    out.release()
    print(f"✅ Successfully saved: {output_filename}")


video_filename = "v_Punch_g04_c04.avi"
video_path = None

for root, dirs, files in os.walk('/kaggle/input'):
    if video_filename in files:
        video_path = os.path.join(root, video_filename)
        break

if video_path:
    print(f"🔍 Found video at: {video_path}")
    
    save_labeled_video(video_path, 'v_Punch_test_result.mp4', model)
else:
    print(f"❌ Error: {video_filename} not found. Check dataset path.")

🔍 Found video at: /kaggle/input/datasets/abdallahwagih/ucf101-videos/test/v_Punch_g04_c04.avi
🎬 Processing Video: v_Punch_g04_c04.avi...
✅ Successfully saved: v_Punch_test_result.mp4
